In [0]:
%run ../../../utils/pipeline_utils

In [0]:
# Databricks notebook source
# =============================================================================
# common/prime_dim_funding.py
# Primes assist_dev.common.dim_funding
#
# Source tables (Silver):
#   silver_aasbs_funding            → base funding package
#   silver_aasbs_funding_amendment  → latest amendment for status/totals
#   silver_aasbs_lu_fund_status     → fund_status_desc decoded
#   silver_aasbs_lu_fund_category   → fund_category_desc decoded
#   silver_aasbs_lu_billing_type    → billing_type_desc decoded
#   silver_aasbs_lu_fund_type       → fund_type_desc decoded
#
# Grain       : One row per funding.id joined to latest funding_amendment
# SCD Type    : 2 — eff_start_dt=NOW, eff_end_dt=NULL, is_current_flag=TRUE
# Idempotent  : YES — TRUNCATE then INSERT
# Dependencies: dim_ia (for ia_sk), dim_agency (for agency_sk)
#
# Source column notes:
#   funding.id                      → funding_id (natural key)
#   funding.ia_id                   → FK to ia → ia_sk
#   funding.fund_status_cd          → fund_status_cd
#   funding.fund_category_cd        → fund_category_cd
#   funding.billing_type_cd         → billing_type_cd (AD=Advance Deposit, RA=Reimb.)
#   funding.latest_funding_amendment_id → FK to funding_amendment.id
#   funding_amendment.id            → funding_amendment_id
#   funding_amendment.fund_amend_status_cd → used for status cross-check
#   funding_amendment.ginv_pop_start_dt/end_dt → period (not on dim_funding)
#   total_funded_amt: sum of loa.loa_dollar_amt for all LOAs on this funding's IAs
#     — not directly stored; computed from loa_ledger in fact notebook.
#     On prime we use funding.funding_num as a proxy identifier only.
#   fund_type_cd: not stored directly on funding; inherited from LOA.
#     Set NULL on prime; enriched via delta refresh from loa join.
#   agency_sk: resolved via ia → ia.activity_address_cd → dim_agency
# =============================================================================

# COMMAND ----------
# MAGIC %run ../utils/pipeline_utils
# COMMAND ----------
dbutils.widgets.text("run_id",   "", "Pipeline Run ID")
dbutils.widgets.text("job_name", "dp1_prime_full", "Job Name")

RUN_ID   = dbutils.widgets.get("run_id")   or "manual-" + get_spark_app_id()
JOB_NAME = dbutils.widgets.get("job_name")

TARGET   = gold("common", "dim_funding")
TASK     = "prime_dim_funding"

S_FUND   = silver("aasbs", "funding")
S_AMEND  = silver("aasbs", "funding_amendment")
S_FSTAT  = silver("aasbs", "lu_fund_status")
S_FCAT   = silver("aasbs", "lu_fund_category")
S_BT     = silver("aasbs", "lu_billing_type")
S_FT     = silver("aasbs", "lu_fund_type")
G_IA     = gold("common", "dim_ia")

print(f"[{TASK}] target={TARGET}")

# COMMAND ----------
start_ts = audit_start(spark, RUN_ID, JOB_NAME, TASK, TARGET,
                        source_schema="aasbs", source_table="funding")

# COMMAND ----------
try:
    truncate_gold(spark, TARGET)

    # funding_sk is GENERATED ALWAYS AS IDENTITY — excluded from INSERT col list.
    spark.sql(f"""
        INSERT INTO {TARGET}
        (
            funding_id, funding_amendment_id,
            fund_status_cd, fund_status_desc,
            fund_category_cd, fund_category_desc,
            billing_type_cd, billing_type_desc,
            fund_type_cd, fund_type_desc,
            ia_sk, agency_sk,
            total_funded_amt, fiscal_year,
            eff_start_dt, eff_end_dt, is_current_flag,
            _gold_created_at, _gold_updated_at, _source_batch_id
        )
        WITH funding_base AS (
            SELECT
                f.id                                AS funding_id,
                f.ia_id,
                f.fund_status_cd,
                f.fund_category_cd,
                f.billing_type_cd,
                -- latest_funding_amendment_id is the FK to the latest amendment
                f.latest_funding_amendment_id       AS funding_amendment_id
            FROM {S_FUND} f
            WHERE NVL(f.is_deleted,FALSE) = FALSE
        ),
        with_amendment AS (
            -- Join to latest amendment for additional metadata.
            -- fund_amend_status_cd cross-checks fund_status_cd.
            -- ginv_pop_start_dt/end_dt define the funding period (not on Gold dim).
            SELECT
                fb.*,
                fa.fund_amend_status_cd,
                fa.ginv_pop_start_dt,
                fa.ginv_pop_end_dt
            FROM funding_base fb
            LEFT JOIN {S_AMEND} fa
                ON fa.id = fb.funding_amendment_id
                AND NVL(fa.is_deleted,FALSE) = FALSE
        ),
        with_decoded AS (
            SELECT
                wa.*,
                COALESCE(fs.description, wa.fund_status_cd)   AS fund_status_desc,
                COALESCE(fc.description, wa.fund_category_cd) AS fund_category_desc,
                COALESCE(bt.description, wa.billing_type_cd)  AS billing_type_desc,
                -- fund_type_cd not on funding row; NULL on prime
                CAST(NULL AS STRING)                           AS fund_type_cd,
                CAST(NULL AS STRING)                           AS fund_type_desc
            FROM with_amendment wa
            LEFT JOIN {S_FSTAT} fs ON wa.fund_status_cd   = fs.cd AND fs.is_deleted = FALSE
            LEFT JOIN {S_FCAT}  fc ON wa.fund_category_cd = fc.cd AND fc.is_deleted = FALSE
            LEFT JOIN {S_BT}    bt ON wa.billing_type_cd  = bt.cd AND bt.is_deleted = FALSE
        ),
        with_ia_sk AS (
            -- Resolve ia_sk from dim_ia using ia_id natural key
            SELECT
                wd.*,
                ia.ia_sk,
                -- agency_sk inherited from dim_ia (servicing_agency_sk)
                ia.servicing_agency_sk              AS agency_sk,
                ia.fiscal_year
            FROM with_decoded wd
            LEFT JOIN {G_IA} ia
                ON wd.ia_id = ia.ia_id
                AND NVL(ia.is_current_flag,FALSE) = TRUE
        )
        SELECT
            funding_id,
            funding_amendment_id,
            fund_status_cd,
            fund_status_desc,
            fund_category_cd,
            fund_category_desc,
            billing_type_cd,
            billing_type_desc,
            fund_type_cd,
            fund_type_desc,
            ia_sk,
            agency_sk,
            -- total_funded_amt: sum of LOA dollar amounts on this funding's IAs.
            -- Computed in the fact notebook from loa_ledger.loa_funded_amt.
            -- NULL on prime; populated via delta refresh.
            CAST(NULL AS DECIMAL(15,2))             AS total_funded_amt,
            fiscal_year,
            -- SCD2 initial values
            CURRENT_TIMESTAMP()                     AS eff_start_dt,
            CAST(NULL AS TIMESTAMP)                 AS eff_end_dt,
            TRUE                                    AS is_current_flag,
            CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP(), '{RUN_ID}'
        FROM with_ia_sk
    """)

    n = row_count(spark, TARGET)
    no_ia = spark.sql(
        f"SELECT COUNT(*) AS n FROM {TARGET} WHERE ia_sk IS NULL"
    ).collect()[0]["n"]
    print(f"  [OK] Inserted {n:,} funding rows ({no_ia} with unresolved ia_sk)")

    audit_success(spark, RUN_ID, TARGET, n, n, start_ts)

except Exception as e:
    audit_fail(spark, RUN_ID, TARGET, str(e), traceback.format_exc(), start_ts)
    raise

# COMMAND ----------
dbutils.notebook.exit("SUCCESS")
